# NumCompute quickstart

An end-to-end tour of the NumCompute toolkit, built with plain Python + NumPy only (no pandas, no scikit-learn).

| Section | Module(s) | What it shows |
|---|---|---|
| 1 | `io` | Reading a CSV, including rows with missing values |
| 2 | `preprocessing`, `pipeline` | Imputer, StandardScaler, OneHotEncoder, Pipeline, FeatureUnion |
| 3 | `sort_search` | stable sort, multi-key sort, top-k, quickselect, binary search |
| 4 | `rank` | Tie handling and percentiles |
| 5 | `stats` | Mean/variance, streaming (Welford) stats, histogram, quantiles |
| 6 | `optim` | Finite-difference gradient and Jacobian |
| 7 | `metrics` | Confusion matrix, accuracy/precision/recall/F1, ROC and AUC |
| 8 | `utils` | Stable softmax and logsumexp |
| 9 | `benchmarking` | Vectorised code vs Python loops, with a performance table |

Run the notebook from top to bottom (**Run All**). The data is `Dataset/Housing.csv` (545 houses).

## 0. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np

# Find the repo root whether the notebook is opened from demo/ or from the root.
ROOT = Path.cwd()
if not (ROOT / "numcompute").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

np.set_printoptions(precision=4, suppress=True, linewidth=110)

HOUSING = ROOT / "Dataset" / "Housing.csv"
print("Repo root:", ROOT)
print("NumPy", np.__version__, "| Python", sys.version.split()[0])

Repo root: d:\Uni\COMP5004\A1\2650_COMP_5004_2_Assignment-2.1-Programming-Task-1-group-Beta
NumPy 2.5.1 | Python 3.14.6


## 1. Reading data with `io.read_csv`

`read_csv` returns `(X, colnames)`. Missing cells (`''`, `NA`, `NaN`) become `NaN` in numeric mode. Text columns are read with `dtype=str`.

In [2]:
from numcompute.io import read_csv, read_csv_chunks

# Numeric columns of the housing file (blank or "NA" cells would become NaN)
X_missing, cols = read_csv(HOUSING, usecols=["price", "area", "bedrooms", "bathrooms"])
print("columns:", cols)
print("shape  :", X_missing.shape)
print(X_missing)
print("NaN count per column:", np.isnan(X_missing).sum(axis=0))

columns: ['price', 'area', 'bedrooms', 'bathrooms']
shape  : (545, 4)
[[13300000.     7420.        4.        2.]
 [12250000.     8960.        4.        4.]
 [12250000.     9960.        3.        2.]
 ...
 [ 1750000.     3620.        2.        1.]
 [ 1750000.     2910.        3.        1.]
 [ 1750000.     3850.        3.        1.]]
NaN count per column: [0 1 3 2]


In [3]:
# Text column: read as strings (dtype=str)
furnishing, _ = read_csv(HOUSING, usecols=["furnishingstatus"], dtype=str)
print(furnishing[:5].ravel())

['furnished' 'furnished' 'semi-furnished' 'furnished' 'furnished']


In [4]:
# The full housing file: numeric columns only
num_cols = ["price", "area", "bedrooms", "bathrooms", "stories", "parking"]
X, cols = read_csv(HOUSING, usecols=num_cols)
print(cols, X.shape)
print(X[:3])

['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'parking'] (545, 6)
[[13300000.     7420.        4.        2.        3.        2.]
 [12250000.     8960.        4.        4.        4.        3.]
 [12250000.     9960.        3.        2.        2.        2.]]


For files too big for memory, `read_csv_chunks` streams the file in fixed-size blocks (used again in section 5).

In [5]:
for i, (chunk, _) in enumerate(read_csv_chunks(HOUSING, chunksize=200, usecols=num_cols)):
    print(f"chunk {i}: {chunk.shape}")

chunk 0: (200, 6)
chunk 1: (200, 6)
chunk 2: (145, 6)


## 2. Preprocessing and pipelines

Every transformer follows the same protocol: `fit(X)` learns statistics, `transform(X)` applies them, `fit_transform(X)` does both.

In [6]:
from numcompute.preprocessing import SimpleImputer, StandardScaler, MinMaxScaler, OneHotEncoder

# Fill the missing values with each column's median
imputer = SimpleImputer(strategy="median")
X_filled = imputer.fit_transform(X_missing)
print("medians learned:", imputer.statistics_)
print("NaNs left:", int(np.isnan(X_filled).sum()))
print(X_filled[:5])

medians learned: [4340000.    4580.       3.       1.]
NaNs left: 0
[[13300000.     7420.        4.        2.]
 [12250000.     8960.        4.        4.]
 [12250000.     9960.        3.        2.]
 [12215000.     7500.        4.        2.]
 [11410000.     7420.        4.        1.]]


In [7]:
# Standardise to zero mean and unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)
print("column means after:", X_scaled.mean(axis=0))
print("column stds  after:", X_scaled.std(axis=0))

column means after: [ 0. -0. -0.  0.]
column stds  after: [1. 1. 1. 1.]


In [8]:
# One-hot encode a text column
enc = OneHotEncoder()
onehot = enc.fit_transform(furnishing)
print("categories:", enc.categories_[0])
print("feature names:", enc.get_feature_names_out(["furnishing"]))
print(onehot[:5])

categories: ['furnished' 'semi-furnished' 'unfurnished']
feature names: ['furnishing_furnished' 'furnishing_semi-furnished' 'furnishing_unfurnished']
[[1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


### Chaining steps with `Pipeline` and joining outputs with `FeatureUnion`

In [9]:
from numcompute.pipeline import Pipeline, FeatureUnion

pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
X_pipe = pipe.fit_transform(X_missing)
print(pipe)
print("same result as doing the steps by hand:", np.allclose(X_pipe, X_scaled))

Pipeline([('impute', SimpleImputer(strategy='median', fill_value=0)), ('scale', StandardScaler(with_mean=True, with_std=True))])
same result as doing the steps by hand: True


In [10]:
union = FeatureUnion([
    ("zscore", StandardScaler()),
    ("minmax", MinMaxScaler()),
])
X_union = union.fit_transform(X_filled)
print("input shape :", X_filled.shape)
print("output shape:", X_union.shape, "(z-scores side by side with min-max values)")

input shape : (545, 4)
output shape: (545, 8) (z-scores side by side with min-max values)


### Every column: numbers, yes/no and text, with missing cells anywhere

The housing file has 13 columns: 6 numeric ones, 6 yes/no ones and `furnishingstatus`. Missing numbers become `NaN`, so `SimpleImputer` can fill them. Missing text cells are read as empty strings, so we fill them with the most common value of their column, then one-hot encode all the text columns together.

In [11]:
text_cols = ["mainroad", "guestroom", "basement", "hotwaterheating",
             "airconditioning", "prefarea", "furnishingstatus"]

X_num, _ = read_csv(HOUSING, usecols=num_cols)                 # blank cells -> NaN
X_txt, _ = read_csv(HOUSING, usecols=text_cols, dtype=str)     # blank cells -> ""

MISSING_TEXT = ["", "NA", "nan", "NAN"]

def fill_text_blanks(column):
    """Replace blank / NA cells of a text column with its most common real value."""
    column = np.char.strip(column)
    blank = np.isin(column, MISSING_TEXT)
    values, counts = np.unique(column[~blank], return_counts=True)
    return np.where(blank, values[np.argmax(counts)], column)

X_txt_filled = np.column_stack([fill_text_blanks(X_txt[:, j]) for j in range(X_txt.shape[1])])

numbers_ready = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(X_num))
encoder = OneHotEncoder()
text_ready = encoder.fit_transform(X_txt_filled)
X_all = np.hstack([numbers_ready, text_ready])

print("missing numeric cells:", int(np.isnan(X_num).sum()))
print("missing text cells   :", int(np.isin(np.char.strip(X_txt), MISSING_TEXT).sum()))
print("feature matrix       :", X_all.shape, "| NaNs left:", int(np.isnan(X_all).sum()))
print("first encoded features:", encoder.get_feature_names_out(text_cols)[:6])

missing numeric cells: 10
missing text cells   : 8
feature matrix       : (545, 21) | NaNs left: 0
first encoded features: ['mainroad_no' 'mainroad_yes' 'guestroom_no' 'guestroom_yes' 'basement_no' 'basement_yes']


## 3. `sort_search`: sorting, top-k, quickselect, binary search

Policy for NaN: sorts put NaN last, `topk` ignores NaN, and `quickselect` / `binary_search` reject NaN with a `ValueError`.

In [12]:
from numcompute.sort_search import stable_sort, multi_key_sort, topk, quickselect, binary_search

# The next sections need complete rows, so keep only the houses with no missing cell
complete = X[~np.isnan(X).any(axis=1)]
print("rows kept:", len(complete), "of", len(X))

price = complete[:, cols.index("price")]
area = complete[:, cols.index("area")]
bedrooms = complete[:, cols.index("bedrooms")]

# Top-k: the 5 most expensive houses, with their row numbers
top_prices, top_rows = topk(price, 5)
print("5 highest prices:", top_prices)
print("row numbers     :", top_rows)

# ...and the 3 cheapest
low_prices, low_rows = topk(price, 3, largest=False)
print("3 lowest prices :", low_prices, "rows", low_rows)

rows kept: 535 of 545
5 highest prices: [13300000. 12250000. 12250000. 12215000. 11410000.]
row numbers     : [0 1 2 3 4]
3 lowest prices : [1750000. 1750000. 1750000.] rows [534 533 532]


In [13]:
# Quickselect finds the k-th smallest without sorting everything
n = price.size
median_qs = quickselect(price, n // 2)
print("median price via quickselect:", median_qs)
print("median via np.median        :", np.median(price))

median price via quickselect: 4305000.0
median via np.median        : 4305000.0


In [14]:
# Binary search on a sorted array: returns (insertion index, found?)
areas_sorted = stable_sort(area)
for target in (3000.0, 6000.0, 7420.0, 99999.0):
    idx, found = binary_search(areas_sorted, target)
    print(f"area {target:>8.0f} -> index {idx:>3}, found={found}")

area     3000 -> index  52, found=True
area     6000 -> index 363, found=True
area     7420 -> index 467, found=True
area    99999 -> index 535, found=False


In [15]:
# Multi-key sort: most bedrooms first, and cheaper houses first when bedrooms tie.
# (the LAST key in `keys` is the primary one)
table = np.column_stack([price, area, bedrooms])
ordered = multi_key_sort(table, keys=[0, 2], ascending=[True, False])
print("price      area   bedrooms")
print(ordered[:6])

price      area   bedrooms
[[3500000.    3600.       6.]
 [6083000.    4300.       6.]
 [1960000.    3420.       5.]
 [3773000.    2520.       5.]
 [3850000.    5300.       5.]
 [4340000.    1905.       5.]]


## 4. `rank`: tie handling and percentiles

Three ways to rank tied values:
- `average`: tied values share the mean of the ranks they cover
- `dense`: tied values share a rank, with no gaps afterwards
- `ordinal`: every value gets a distinct rank (ties broken by position)

In [16]:
from numcompute.rank import rank, percentile

scores = np.array([10, 20, 20, 30, 20, 5])
print("scores :", scores)
for method in ("average", "dense", "ordinal"):
    print(f"{method:>8}:", rank(scores, method=method))

scores : [10 20 20 30 20  5]
 average: [2. 4. 4. 6. 4. 1.]
   dense: [2. 3. 3. 4. 3. 1.]
 ordinal: [2. 3. 4. 6. 5. 1.]


In [17]:
# Percentiles of the house prices (they match NumPy's answer)
qs = [10, 25, 50, 75, 90]
mine = percentile(price, qs)
ref = np.percentile(price, qs)
print("percentiles:", qs)
print("NumCompute :", mine)
print("np         :", ref)
print("match:", np.allclose(mine, ref))

percentiles: [10, 25, 50, 75, 90]
NumCompute : [2835000. 3430000. 4305000. 5600000. 7056000.]
np         : [2835000. 3430000. 4305000. 5600000. 7056000.]
match: True


In [18]:
# NaN values are ignored by percentile
with_nan = np.array([1.0, 2.0, np.nan, 4.0, 5.0])
print("median ignoring NaN:", percentile(with_nan, 50))

median ignoring NaN: 3.0


## 5. `stats`: descriptive statistics

`describe` is NaN-aware, `StreamingStats` uses Welford's algorithm so it never needs all the data at once, and `histogram` / `quantile` summarise the distribution.

In [19]:
from numcompute.stats import describe, StreamingStats, histogram, quantile

summary = describe(X_missing)          # NaN-aware: ignores missing cells
for name, vals in summary.items():
    print(f"{name:>6}:", vals)

  mean: [4766729.2477    5135.7445       2.9649       1.2836]
median: [4340000.    4580.       3.       1.]
   std: [1868722.8281    2142.4721       0.7394       0.5011]
   min: [1750000.    1650.       1.       1.]
   max: [13300000.    16200.        6.        4.]


In [20]:
# Streaming mean/variance: one accumulator per chunk, then merge
# (cells that are missing are skipped, as np.nanmean / np.nanvar do)
all_prices = X[:, cols.index("price")]
total = StreamingStats()
for chunk, _ in read_csv_chunks(HOUSING, chunksize=100, usecols=["price"]):
    part = StreamingStats()
    for value in chunk[:, 0]:
        if not np.isnan(value):
            part.update(value)
    total.merge(part)

print("streaming mean    :", total.mean)
print("np.nanmean        :", np.nanmean(all_prices))
print("streaming variance:", total.variance)
print("np.nanvar         :", np.nanvar(all_prices))
print("count             :", total.count)

streaming mean    : 4766729.247706423
np.nanmean        : 4766729.247706422
streaming variance: 3492125008378.704
np.nanvar         : 3492125008378.7007
count             : 545


In [21]:
# Histogram of house prices, drawn with text bars
counts, edges = histogram(price, bins=8)
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"{lo/1e6:5.1f}M - {hi/1e6:5.1f}M | {'#' * (c // 8)} {c}")

  1.8M -   3.2M | ############ 100
  3.2M -   4.6M | ########################## 212
  4.6M -   6.1M | ############## 119
  6.1M -   7.5M | ######## 65
  7.5M -   9.0M | ### 25
  9.0M -  10.4M | # 8
 10.4M -  11.9M |  2
 11.9M -  13.3M |  4


In [22]:
print("quantiles 0.25 / 0.5 / 0.75:", quantile(price, [0.25, 0.5, 0.75]))

quantiles 0.25 / 0.5 / 0.75: [3430000. 4305000. 5600000.]


## 6. `optim`: finite-difference gradient and Jacobian

`grad(f, x)` estimates the gradient of a scalar function. `jacobian(F, x)` estimates the matrix of partial derivatives of a vector function. Both use central differences by default, and we compare them with the exact (hand-derived) answers.

In [23]:
from numcompute.optim import grad, jacobian

# Scalar function: f(x, y) = x^2 * y + sin(y)
def f(v):
    x, y = v
    return x**2 * y + np.sin(y)

point = np.array([1.5, 2.0])
numeric = grad(f, point)
exact = np.array([2 * point[0] * point[1],              # df/dx = 2xy
                  point[0] ** 2 + np.cos(point[1])])    # df/dy = x^2 + cos(y)
print("numeric gradient:", numeric)
print("exact gradient  :", exact)
print("max error       :", np.abs(numeric - exact).max())

numeric gradient: [6.     1.8339]
exact gradient  : [6.     1.8339]
max error       : 4.9183768169314135e-11


In [24]:
# Vector function: F(x, y) = [x*y, x + y^2, sin(x)]
def F(v):
    x, y = v
    return np.array([x * y, x + y**2, np.sin(x)])

point = np.array([0.5, 2.0])
J = jacobian(F, point)
J_exact = np.array([[point[1], point[0]],
                    [1.0, 2 * point[1]],
                    [np.cos(point[0]), 0.0]])
print("Jacobian shape:", J.shape, "(3 outputs x 2 inputs)")
print(J)
print("max error vs exact:", np.abs(J - J_exact).max())

Jacobian shape: (3, 2) (3 outputs x 2 inputs)
[[2.     0.5   ]
 [1.     4.    ]
 [0.8776 0.    ]]
max error vs exact: 3.78578279836006e-11


In [25]:
# Central vs forward differences: central is more accurate for the same step
point = np.array([1.5, 2.0])
for method in ("forward", "central"):
    err = np.abs(grad(f, point, h=1e-4, method=method) - exact).max()
    print(f"{method:>8} (h=1e-4): max error = {err:.2e}")

 forward (h=1e-4): max error = 3.00e-04
 central (h=1e-4): max error = 2.77e-09


## 7. `metrics`: evaluating a classifier

A tiny rule-based classifier on the housing data: predict "expensive" (price above the median) from the house area alone, then score it with our metrics.

In [26]:
from numcompute.metrics import (confusion_matrix, accuracy, precision, recall, f1,
                                mse, roc_curve, auc)

y_true = (price > np.median(price)).astype(int)
score = (area - area.mean()) / area.std()       # bigger house -> more likely expensive
y_pred = (score > 0).astype(int)

cm, labels = confusion_matrix(y_true, y_pred)
print("class labels:", labels)
print("confusion matrix (rows = true, cols = predicted):")
print(cm)
print("accuracy :", round(accuracy(y_true, y_pred), 4))
print("precision:", round(precision(y_true, y_pred), 4))
print("recall   :", round(recall(y_true, y_pred), 4))
print("f1       :", round(f1(y_true, y_pred), 4))

class labels: [0 1]
confusion matrix (rows = true, cols = predicted):
[[218  51]
 [ 94 172]]
accuracy : 0.729
precision: 0.7713
recall   : 0.6466
f1       : 0.7035


In [27]:
fpr, tpr, thresholds = roc_curve(y_true, score)
print("ROC points:", len(fpr))
print("AUC:", round(auc(fpr, tpr), 4), "(0.5 = random guessing, 1.0 = perfect)")

# Regression error: predict price from area with a straight line
A_design = np.column_stack([area, np.ones_like(area)])
coef, *_ = np.linalg.lstsq(A_design, price, rcond=None)
print("MSE of the line fit:", f"{mse(price, A_design @ coef):.4e}")

ROC points: 281
AUC: 0.7953 (0.5 = random guessing, 1.0 = perfect)
MSE of the line fit: 2.4018e+12


## 8. `utils`: numerically stable helpers

A naive softmax overflows on large inputs. NumCompute subtracts the maximum first.

In [28]:
from numcompute.utils import softmax, logsumexp

big = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(all="ignore"):
    naive = np.exp(big) / np.exp(big).sum()
print("naive softmax :", naive, "  <- overflow gives NaN")
print("stable softmax:", softmax(big))
print("logsumexp     :", logsumexp(big))

naive softmax : [nan nan nan]   <- overflow gives NaN
stable softmax: [0.09   0.2447 0.6652]
logsumexp     : 1002.4076059644444


## 9. Benchmarks: vectorised vs Python loops

`benchmark/run_benchmarks.py` times seven NumCompute operations on the Housing data against hand-written plain-Python loops that compute the same result. `compare()` first checks that both versions return the same answer, then reports the speedup. Running `python -m benchmark.run_benchmarks` from the repo root prints the same table. The next cell takes a few seconds.

In [ ]:
from benchmark.run_benchmarks import DATA_PATH, REPEATS, build_cases, load_housing
from numcompute.benchmarking import compare, format_results_table
from IPython.display import Markdown, display

numeric, features, has_ac = load_housing(DATA_PATH)
results = []
for label, vectorised, loop in build_cases(numeric, features, has_ac):
    result = compare(vectorised, loop, repeats=REPEATS)   # checks both versions agree first
    result["name"] = label
    results.append(result)

display(Markdown(format_results_table(results)))

In [ ]:
import os
import platform

print("Environment:")
print(f"- Python {platform.python_version()}, NumPy {np.__version__}")
print(f"- {platform.system()} {platform.release()} ({platform.machine()}), {os.cpu_count()} logical cores")
print(f"- Each time is the mean of {REPEATS} runs after 1 warm-up run, on {len(numeric)} houses")

## Summary

- `io` read data with missing values; `preprocessing` and `pipeline` cleaned and scaled it.
- `sort_search` and `rank` handled ordering, selection and ties, and agreed with NumPy.
- `stats`, `optim` and `metrics` summarised the data, checked gradients against exact answers, and scored a classifier.
- The benchmark table shows large speedups where the loop does pairwise work (distances, ranking, ROC), little or no gain on quick one-pass summaries of 545 rows, and the same answers from both versions.